# Tutorial
## Fine Tuning Transformers from HuggingFace
## References:
1. Fine Tuning on SQuAD: https://huggingface.co/transformers/v3.1.0/custom_datasets.html?highlight=forsequenceclassification#qa-squad
2. Evaluation on QA Datasets: https://qa.fastforwardlabs.com/no%20answer/null%20threshold/bert/distilbert/exact%20match/f1/robust%20predictions/2020/06/09/Evaluating_BERT_on_SQuAD.html

In [ ]:
!mkdir squad
!wget https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json -O squad/train-v2.0.json
!wget https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json -O squad/dev-v2.0.json

## SQuAD 2.0 Dataset
We will be using the SQuAD 2.0 Dataset which is based on the Question answering task as mentioned in class. We first visualize a question from the dataset.

In [ ]:
import json

with open('squad/train-v2.0.json', 'rb') as f:
    raw_dataset = json.load(f)['data']

item = raw_dataset[0]
paragraph = item['paragraphs'][0]

qa = paragraph['qas'][0]
answer = qa['answers']

print('Context Paragraph')
print(paragraph['context'])
print()
print('Question')
print(qa['question'])
print()
print('Answer')
print(answer)

Context Paragraph
Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny's Child. Managed by her father, Mathew Knowles, the group became one of the world's best-selling girl groups of all time. Their hiatus saw the release of Beyoncé's debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".

Question
When did Beyonce start becoming popular?

Answer
[{'text': 'in the late 1990s', 'answer_start': 269}]


## Data Pre-Processing
Reference: https://huggingface.co/transformers/v3.1.0/custom_datasets.html?highlight=forsequenceclassification#qa-squad

Here we start off by preprocessing the dataset. 
1. We start off by tokenizing the all context paragraphs and questions in the dataset. 
2. We also keep track of the index for which the answer starts and ends within the context paragraph.
3. See what the tokenizer returns here: https://huggingface.co/docs/transformers/v4.24.0/en/main_classes/tokenizer#transformers.PreTrainedTokenizer.__call__
4. We convert the indexes of these answers to positions in the token sequence with `tokens.char_to_token`.
5. Finally we update the tokens data structure with the start and end indexes of the answer.

Note that these conversion are required as BPE encoding is used so the number of tokens is not the same as the number of words after tokenization.

In [ ]:
from transformers import DistilBertTokenizerFast

def process_data_filtered(dataset_path):
    with open(dataset_path, 'rb') as f:
        raw_dataset = json.load(f)

    dataset = []
    for item in raw_dataset['data']:
        for paragraph in item['paragraphs']:
            for qa in paragraph['qas']:
                for answer in qa['answers']:
                    if len(answer['text']) != 0:
                        answer_start = answer['answer_start']
                        answer_end = answer_start + len(answer['text'])
                        answer['answer_end'] = answer_end
                        dataset.append({'context': paragraph['context'], 'question': qa['question'], 'answer': answer})

    contexts = [item['context'] for item in dataset]
    questions = [item['question'] for item in dataset]
    
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    tokens = tokenizer(contexts, questions, truncation=True, padding=True)

    start_indexs = []
    end_indexs = []
    for idx, data in enumerate(dataset):
        context = data['context']
        answer = data['answer']

        start_index = tokens.char_to_token(idx, answer['answer_start'])
        end_index = tokens.char_to_token(idx, answer['answer_end']-1)

        if start_index is None:
            start_index = tokenizer.model_max_length
        if end_index is None:
            end_index = tokenizer.model_max_length

        start_indexs.append(start_index)
        end_indexs.append(end_index)
    tokens.update({'start_index': start_indexs, 'end_index': end_indexs})

    print('Number of samples in qa dataset: {}'.format(len(dataset)))
    return dataset, tokens

processed_dataset, tokens = process_data_filtered('./squad/train-v2.0.json')
print(tokens[0])
print()
print('Input Tokens')
print(tokens['input_ids'][0])
print()
print('Attention Mask')
print(tokens['attention_mask'][0])
print()
print('Answer Start Index')
print(tokens['start_index'][0])
print()
print('Answer End Index')
print(tokens['end_index'][0])

Number of samples in qa dataset: 86821
Encoding(num_tokens=512, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

Input Tokens
[101, 20773, 21025, 19358, 22815, 1011, 5708, 1006, 1013, 12170, 23432, 29715, 3501, 29678, 12325, 29685, 1013, 10506, 1011, 10930, 2078, 1011, 2360, 1007, 1006, 2141, 2244, 1018, 1010, 3261, 1007, 2003, 2019, 2137, 3220, 1010, 6009, 1010, 2501, 3135, 1998, 3883, 1012, 2141, 1998, 2992, 1999, 5395, 1010, 3146, 1010, 2016, 2864, 1999, 2536, 4823, 1998, 5613, 6479, 2004, 1037, 2775, 1010, 1998, 3123, 2000, 4476, 1999, 1996, 2397, 4134, 2004, 2599, 3220, 1997, 1054, 1004, 1038, 2611, 1011, 2177, 10461, 1005, 1055, 2775, 1012, 3266, 2011, 2014, 2269, 1010, 25436, 22815, 1010, 1996, 2177, 2150, 2028, 1997, 1996, 2088, 1005, 1055, 2190, 1011, 4855, 2611, 2967, 1997, 2035, 2051, 1012, 2037, 14221, 2387, 1996, 2713, 1997, 20773, 1005, 1055, 2834, 2201, 1010, 20754, 1999, 2293, 1006, 2494, 1007, 1010, 2029, 2511, 2014, 2004,

## Dataset Loader
We use pytorch lightning for this tutorial.
1. We define a simple Dataset Loader to return the all items of a token object at the index idx.
2. We define a simple PyTorch Lightning Module that returns the Data Loader for the Dataset.

In [ ]:
import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class QADataset(Dataset):
    def __init__(self, dataset, tokens):
        self.dataset = dataset
        self.tokens = tokens

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.tokens.items()}

class QADataModule(pl.LightningDataModule):
    def __init__(self, dataset, tokens, batch_size):
        super().__init__()
        self.dataset = dataset
        self.tokens = tokens
        self.batch_size = batch_size

    def setup(self, stage=None):
        self.train_data = QADataset(self.dataset, self.tokens)

    def train_dataloader(self):
        return DataLoader(self.train_data, batch_size=self.batch_size, num_workers=0)
    
batch_size = 32
data = QADataModule(processed_dataset, tokens, batch_size)

## Lightning Module for Model
Here we define a PyTorch Lightning module for the model for training. This is essentially a trainer so we don't have to write our own for loops in training as we have done in the past. This is done automatically in PL. We just have to define some functions:
1. configure_optimizers: The optimizer we will be using for training.
2. forward: The forward function for the model.
3. training_step:Returns the loss function that should be optimized.
4. on_epoch_end: Function that will be called at the end of every epoch.

In [ ]:
from transformers import AdamW
from torch.utils.tensorboard import SummaryWriter

class QAModel(pl.LightningModule):
    def __init__(self, model, prefix, save_directory):
        super().__init__()
        self.counter = 0
        self.model = model
        self.prefix = prefix
        self.save_directory = save_directory
        self.writer = SummaryWriter('runs_{}'.format(self.prefix))

    def get_progress_bar_dict(self):
        items = super().get_progress_bar_dict()
        items.pop("v_num", None)
        return items

    def configure_optimizers(self):
        return AdamW(self.model.parameters(), lr=5e-5)

    def forward(self, x):
        input_ids = x['input_ids']
        attention_mask = x['attention_mask']
        start_index = x['start_index']
        end_index = x['end_index']

        outputs = self.model(input_ids, attention_mask=attention_mask, start_positions=start_index, end_positions=end_index)
        return outputs

    def training_step(self, batch, batch_idx):
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        start_index = batch['start_index']
        end_index = batch['end_index']

        outputs = self.model(input_ids, attention_mask=attention_mask, start_positions=start_index, end_positions=end_index)
        loss = outputs[0]
        self.writer.add_scalar('Loss/train', loss.item(), self.counter)
        self.counter += 1
        return loss

    def on_epoch_end(self):
        torch.save(self.model.state_dict(), self.save_directory + '/{}{}.pth'.format(self.prefix, self.current_epoch))

## Define the Model
We will be using DistilBert for this tutorial. This is mainly due to its small size compared to the original BERT. It is trained using a transfer learning called distilled learning which distils the weights of BERT to a smaller transformer network: https://arxiv.org/abs/1910.01108. It is 40% smaller than the original BERT while retaining 97% of its language understanding capabilities, which makes it easier to train on GPUs with less memory.

There are usually two ways to initialize the transformer model from hugging face:
1. Defining the model with your own hyperparameters: https://huggingface.co/docs/transformers/v4.24.0/en/model_doc/distilbert#transformers.DistilBertConfig
2. Depending on the task you want to do:
https://huggingface.co/docs/transformers/v4.24.0/en/model_doc/distilbert#distilbert

In [ ]:
import os
from transformers import DistilBertForQuestionAnswering

save_directory = './checkpoints'
if not os.path.exists(save_directory):
    os.makedirs(save_directory)

distilbert = DistilBertForQuestionAnswering.from_pretrained('distilbert-base-uncased')
model = QAModel(distilbert, 'squad', save_directory)

Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertForQuestionAnswering: ['vocab_projector.weight', 'vocab_transform.bias', 'vocab_transform.weight', 'vocab_layer_norm.bias', 'vocab_projector.bias', 'vocab_layer_norm.weight']
- This IS expected if you are initializing DistilBertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this mode

## Start Training
With the LightModule we do not have to write our own for loops for the training and we just have to call the `.fit()` function with the LightningDataLoader which will return the dataloader for the training dataloader.

In [ ]:
epochs = 4

trainer = pl.Trainer(gpus=1, fast_dev_run=False, max_epochs=epochs)
trainer.fit(model, data)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Missing logger folder: /home/rexlee/Projects/JerichoKG/lightning_logs


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type                           | Params
---------------------------------------------------------
0 | model | DistilBertForQuestionAnswering | 66.4 M
---------------------------------------------------------
66.4 M    Trainable params
0         Non-trainable params
66.4 M    Total params
265.458   Total estimated model params size (MB)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Training: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=4` reached.


## Helper Functions
Reference: https://qa.fastforwardlabs.com/no%20answer/null%20threshold/bert/distilbert/exact%20match/f1/robust%20predictions/2020/06/09/Evaluating_BERT_on_SQuAD.html
Here we use the helper functions from the guide above for calculating the EM and F1 scores for QA, as well as define the function to extract answers from the output of DistilBert. See the reference for more details

In [ ]:
import torch.nn as nn

def normalize_text(s):
    """
    From https://qa.fastforwardlabs.com/no%20answer/null%20threshold/bert/distilbert/exact%20match/f1/robust%20predictions/2020/06/09/Evaluating_BERT_on_SQuAD.html
    """
    """Removing articles and punctuation, and standardizing whitespace are all typical text processing steps."""
    import string, re

    def remove_articles(text):
        regex = re.compile(r"\b(a|an|the)\b", re.UNICODE)
        return re.sub(regex, " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

def compute_exact_match(prediction, truth):
    """
    From: https://qa.fastforwardlabs.com/no%20answer/null%20threshold/bert/distilbert/exact%20match/f1/robust%20predictions/2020/06/09/Evaluating_BERT_on_SQuAD.html
    """
    return int(normalize_text(prediction) == normalize_text(truth))

def compute_f1(prediction, truth):
    """
    From: https://qa.fastforwardlabs.com/no%20answer/null%20threshold/bert/distilbert/exact%20match/f1/robust%20predictions/2020/06/09/Evaluating_BERT_on_SQuAD.html
    """
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(truth).split()
    
    # if either the prediction or the truth is no-answer then f1 = 1 if they agree, 0 otherwise
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)
    
    common_tokens = set(pred_tokens) & set(truth_tokens)
    
    # if there are no common tokens then f1 = 0
    if len(common_tokens) == 0:
        return 0
    
    prec = len(common_tokens) / len(pred_tokens)
    rec = len(common_tokens) / len(truth_tokens)
    
    return 2 * (prec * rec) / (prec + rec)

def get_prediction(input, model):
    """
    Modified from: https://qa.fastforwardlabs.com/no%20answer/null%20threshold/bert/distilbert/exact%20match/f1/robust%20predictions/2020/06/09/Evaluating_BERT_on_SQuAD.html
    """
    input_ids = input['input_ids'].cuda()
    attention_mask = input['attention_mask'].cuda()
    outputs = model(input_ids, attention_mask=attention_mask)

    answer_start = torch.argmax(outputs[0],-1)
    answer_end = torch.argmax(outputs[1],-1) + 1

    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    answers = []
    for i in range(answer_start.shape[0]):
        answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(input_ids[i][answer_start[i]:answer_end[i]]))
        answers.append(answer)

    return answers

## Evaluation
Finally we evaluate the SQuAD dataset using the dev split using the EM and F1 scores.

In [ ]:
def process_data_val(dataset_path):
    with open(dataset_path, 'rb') as f:
        raw_dataset = json.load(f)

    contexts = []
    questions = []
    dataset = []
    for item in raw_dataset['data']:
        for paragraph in item['paragraphs']:
            for qa in paragraph['qas']:
                if len(qa['answers']) == 0:
                    dataset.append({'context': paragraph['context'], 'question': qa['question'], 'answer': None})
                else:
                    dataset.append({'context': paragraph['context'], 'question': qa['question'], 'answer': qa['answers']})

    contexts = [item['context'] for item in dataset]
    questions = [item['question'] for item in dataset]
    
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    tokens = tokenizer(contexts, questions, truncation=True, padding=True)

    print('Number of samples in qa dataset: {}'.format(len(dataset)))
    return dataset, tokens

In [ ]:
from tqdm import tqdm

batch_size = 64
num_workers = 0

processed_dataset, tokens = process_data_val('./squad/dev-v2.0.json')
dataset = QADataset(processed_dataset, tokens)
dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=num_workers, drop_last=False, shuffle=False)

model = DistilBertForQuestionAnswering.from_pretrained('distilbert-base-uncased').cuda()
model.load_state_dict(torch.load('./checkpoints/squad3.pth'))
model.eval()

total_em = 0
total_f1 = 0
counter = 0

squad_predictions = []

with torch.no_grad():
    for idx, batch in enumerate(tqdm(dataloader)):
        predictions = get_prediction(batch, model)

        for offset, pred in enumerate(predictions):
            if not processed_dataset[idx*batch_size+offset]['answer']:
                answers = ['']
            else:
                answers = [answer['text'] for answer in processed_dataset[idx*batch_size+offset]['answer']]

            item = {}
            item['question'] = processed_dataset[idx*batch_size+offset]['question']
            item['gt_answers'] = answers
            item['pred_answer'] = predictions[offset]
            squad_predictions.append(item)

            em_score = max((compute_exact_match(pred, answer)) for answer in answers)
            f1_score = max((compute_f1(pred, answer)) for answer in answers)

            total_em += em_score
            total_f1 += f1_score
            counter += 1


print('SQuAD Results')
print('EM: {}'.format(total_em/counter))
print('F1: {}'.format(total_f1/counter))
print()

Number of samples in qa dataset: 11873


Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertForQuestionAnswering: ['vocab_projector.weight', 'vocab_transform.bias', 'vocab_transform.weight', 'vocab_layer_norm.bias', 'vocab_projector.bias', 'vocab_layer_norm.weight']
- This IS expected if you are initializing DistilBertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this mode

100%|█████████████████████████████████████████████████████████████████████████████████| 186/186 [01:02<00:00,  2.96it/s]

SQuAD Results
EM: 0.3361408237176788
F1: 0.3924655049769018



## Printing Prediction Results

In [ ]:
print('Question:', squad_predictions[0]['question'])
print('GT Answer:', squad_predictions[0]['gt_answers'])
print('Pred Answer:', squad_predictions[0]['pred_answer'])
print()
print('Question:', squad_predictions[1]['question'])
print('GT Answer:', squad_predictions[1]['gt_answers'])
print('Pred Answer:', squad_predictions[1]['pred_answer'])
print()
print('Question:', squad_predictions[2]['question'])
print('GT Answer:', squad_predictions[2]['gt_answers'])
print('Pred Answer:', squad_predictions[2]['pred_answer'])
print()
print('Question:', squad_predictions[3]['question'])
print('GT Answer:', squad_predictions[3]['gt_answers'])
print('Pred Answer:', squad_predictions[3]['pred_answer'])
print()
print('Question:', squad_predictions[4]['question'])
print('GT Answer:', squad_predictions[4]['gt_answers'])
print('Pred Answer:', squad_predictions[4]['pred_answer'])

Question: In what country is Normandy located?
GT Answer: ['France', 'France', 'France', 'France']
Pred Answer: france

Question: When were the Normans in Normandy?
GT Answer: ['10th and 11th centuries', 'in the 10th and 11th centuries', '10th and 11th centuries', '10th and 11th centuries']
Pred Answer: 10th and 11th centuries

Question: From which countries did the Norse originate?
GT Answer: ['Denmark, Iceland and Norway', 'Denmark, Iceland and Norway', 'Denmark, Iceland and Norway', 'Denmark, Iceland and Norway']
Pred Answer: denmark, iceland and norway

Question: Who was the Norse leader?
GT Answer: ['Rollo', 'Rollo', 'Rollo', 'Rollo']
Pred Answer: rollo

Question: What century did the Normans first gain their separate identity?
GT Answer: ['10th century', 'the first half of the 10th century', '10th', '10th']
Pred Answer: 10th
